# 직접 해보기 08 · 두 관측소의 떨림 겹쳐 보기

GWOSC가 공개한 GW150914 그림 1의 파형(핸퍼드 H, 리빙스턴 L)과 NANOGrav 15년 교육용 요약을 읽는다. 파형에는 35–350 Hz 통과 필터와 장치 선 잡음 제거 필터가 **이미 적용**되어 있다(원시 기록이 아님). 같은 계산이 `analysis.py`에도 들어 있다.

In [ ]:
import analysis as A
H, L = A.wave('H'), A.wave('L')
print(len(H), len(L), '행 · 시각', L[0][0], '–', L[-1][0])
print({n: A.sha8(n) for n in A.FILES})

## 1. 조정 전, 따로 본다
matplotlib이 없으면 `pip install matplotlib`.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
ax[0].plot(*zip(*H), color='k', lw=1); ax[0].set_ylabel('H')
ax[1].plot(*zip(*L), color='#7D5A9E', lw=1); ax[1].set_ylabel('L'); ax[1].set_xlabel('time (s)')
for a in ax: a.axvspan(.30, .45, color='#EFEAF3', zorder=0)
plt.show()

## 2. 논문 조건: H를 6.9 ms 앞당기고 부호를 뒤집는다

In [ ]:
r, n = A.pearson(H, L, shift_ms=6.9, flip=True, start=.30, end=.45)
print('r =', round(r, 3), '· 행', n)
sel = [(t, v) for t, v in L if .30 <= t <= .45]
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot([t for t, _ in sel], [v for _, v in sel], color='#7D5A9E', label='L')
ax.plot([t for t, _ in sel], [-A.interp(H, t + .0069) for t, _ in sel], color='k', lw=1, label='H shifted & flipped')
ax.legend(); plt.show()

## 3. 조건을 하나씩 끈다

In [ ]:
for name, kw in [('논문 조건', dict(shift_ms=6.9, flip=True)), ('부호만 반전', dict(shift_ms=0, flip=True)),
                 ('이동·반전 없음', dict(shift_ms=0, flip=False)), ('이동만', dict(shift_ms=6.9, flip=False)),
                 ('구간 0.25–0.46', dict(shift_ms=6.9, flip=True, start=.25, end=.46))]:
    print(f'{name:<14} r = {A.pearson(H, L, **kw)[0]:+.3f}')

## 4. 이동량에 따른 상관계수

In [ ]:
D = [i * 0.2 - 15 for i in range(151)]
R = [A.pearson(H, L, d, True)[0] for d in D]
best = max(zip(R, D)); print('최대 r', round(best[0], 3), '이동', round(best[1], 1), 'ms')
plt.plot(D, R, color='k'); plt.axvline(6.9, color='#D9733D'); plt.axvline(best[1], color='gray', ls=':')
plt.xlabel('H shift (ms)'); plt.ylabel('r'); plt.show()

## 5. 구간을 좁혀 본다
다음 걸음: 합병 직전 0.40–0.43초로 좁혀 상관계수를 다시 재고 구간·이동량·결과를 세 줄로 기록한다.

In [ ]:
print(A.pearson(H, L, 6.9, True, .40, .43))
print(A.best_shift(H, L, True, .40, .43))

## 6. NANOGrav 교육용 요약 (값 ÷ 오차)
막대 수를 바꿔도 중앙값은 같다. 이것은 검출 시그마가 아니고, 10,000행은 독립된 사건이 아니다.

In [ ]:
rows = A.pta()
for j, name in enumerate(A.COMPONENTS):
    col = [r[j] for r in rows]
    print(name, [round(A.quantile(col, p), 2) for p in (.16, .5, .84)])
hd = [r[2] for r in rows]
for bins in (10, 40, 90):
    plt.hist(hd, bins=bins, range=(-2, 7), alpha=.4)
plt.axvline(A.quantile(hd, .5), color='k'); plt.xlabel('A / A_err (HD)'); plt.show()